# AI analyzing code for vulnerabilities

# Coding problem, 1.5.b: AI analyzing code for vulnerabilities

**Skill:** Detect Attacks, Collaborate · **Maps to:** 1.5.A · **Language:** Python 3 (pytest)

## Problem
Build the pattern-matching core of an AI code security scanner. Given a string of Python source
code, return a sorted list of vulnerability tags for every dangerous pattern found. This is the
rule-evaluation layer that tools like Bandit and Semgrep implement under the hood.

## Function contract
Implement `solve(code: str) -> list[str]` in the code editor (see the Reference section below).
Return a sorted, de-duplicated list of finding tag strings.

## Input
A single string, the entire source code to scan. May be multi-line or empty.

## Output
A sorted `list[str]` of vulnerability tags drawn from:
- `"vuln:eval"`
- `"vuln:hardcoded_password"`
- `"vuln:md5"`
- `"vuln:os_system"`
- `"vuln:pickle_loads"`
- `"vuln:subprocess_shell"`

## Examples
```
'result = eval(user_input)'
    -> ["vuln:eval"]

'os.system("ls " + path)\nsubprocess.run(cmd, shell=True)'
    -> ["vuln:os_system", "vuln:subprocess_shell"]

'import hashlib\nhashlib.md5(data)\npickle.loads(blob)'
    -> ["vuln:md5", "vuln:pickle_loads"]

'x = 1 + 2'
    -> []
```

## Constraints
- Deterministic, offline, standard library only.
- `password =` matching is case-insensitive; all other patterns are case-sensitive.
- Must pass all cases in the tests (visible + hidden).

## Reference
Any implementation that satisfies this interface can be swapped in (reference, student, or
composite) and must pass the tests unchanged.

### Entry point
```python
def solve(code: str) -> list[str]
```

### Input (`code`)
A single string containing Python source code to scan. May be multi-line. May be empty.

### Output (`result`)
A **sorted, de-duplicated** `list[str]` of vulnerability finding tags. Return `[]` if no
patterns are found.

### Pattern rule set (case-sensitive substring search unless noted)

| Tag returned                   | Trigger substring         | Vulnerability class                      |
|--------------------------------|--------------------------|------------------------------------------|
| `"vuln:eval"`                  | `eval(`                  | Code injection via eval                  |
| `"vuln:os_system"`             | `os.system(`             | OS command injection                     |
| `"vuln:subprocess_shell"`      | `shell=True`             | OS command injection via subprocess      |
| `"vuln:md5"`                   | `md5(`                   | Broken cryptographic hash (MD5)          |
| `"vuln:pickle_loads"`          | `pickle.loads(`          | Unsafe deserialization                   |
| `"vuln:hardcoded_password"`    | `password =`             | Hardcoded credential in source           |

Notes:
- `password =` matching is case-insensitive (matches `Password =`, `PASSWORD =`, etc.).
- All other patterns are case-sensitive (Python identifiers are case-sensitive).
- A single line may trigger multiple tags if multiple patterns are present.
- Comments are NOT excluded, a dangerous call inside a comment still triggers the tag.
  (In practice, AST-based tools skip comments; here we use substring scan for transparency.)

### Guarantees / rules
- Pure function, deterministic, no I/O, no network. Standard library only.
- Output always sorted ascending (lexicographic) and de-duplicated.
- `solve("")` returns `[]`.

### Acceptance
Your code passes when every test below shows a checkmark (some are hidden until you run them).


In [ ]:
# Your implementation
"""Starter code for 1.5.b. Student completes the TODO.
This file intentionally FAILS the tests until implemented."""
from typing import List


def solve(code: str) -> List[str]:
    """Return sorted, de-duplicated vulnerability tags for dangerous patterns in Python source code.

    Args:
        code: a string of Python source code to scan.
    Returns:
        Sorted list of finding tags such as "vuln:eval", "vuln:md5", etc.
        See the Reference section below for the full pattern rule set.
    """
    # TODO: search `code` for each dangerous pattern from the Reference section below.
    #       Note: "password =" matching is case-insensitive; others are case-sensitive.
    #       Return a sorted, de-duplicated list of finding tags.
    raise NotImplementedError("implement solve() — see the problem statement above")


## Self-check tests
Run the cell below to check your `solve` implementation above.

In [ ]:
import __main__ as impl  # the functions defined above

import os


# ---- visible cases -------------------------------------------------------
def test_eval_only():
    assert impl.solve("result = eval(user_input)") == ["vuln:eval"]


def test_os_system_and_shell_true():
    code = 'os.system("ls " + path)\nsubprocess.run(cmd, shell=True)'
    assert impl.solve(code) == ["vuln:os_system", "vuln:subprocess_shell"]


def test_md5_and_pickle():
    code = "import hashlib\nhashlib.md5(data)\npickle.loads(blob)"
    assert impl.solve(code) == ["vuln:md5", "vuln:pickle_loads"]


def test_clean_code():
    assert impl.solve("x = 1 + 2\nprint(x)") == []


# ---- hidden / edge cases -------------------------------------------------
def test_empty_string():
    assert impl.solve("") == []


def test_hardcoded_password_lowercase():
    assert impl.solve('password = "secret123"') == ["vuln:hardcoded_password"]


def test_hardcoded_password_mixed_case():
    assert impl.solve('Password = "abc"') == ["vuln:hardcoded_password"]


def test_hardcoded_password_uppercase():
    assert impl.solve('PASSWORD = "abc"') == ["vuln:hardcoded_password"]


def test_all_patterns():
    code = (
        "eval(x)\n"
        "os.system(cmd)\n"
        "subprocess.run(args, shell=True)\n"
        "hashlib.md5(data)\n"
        "pickle.loads(blob)\n"
        'password = "pw"\n'
    )
    assert impl.solve(code) == [
        "vuln:eval",
        "vuln:hardcoded_password",
        "vuln:md5",
        "vuln:os_system",
        "vuln:pickle_loads",
        "vuln:subprocess_shell",
    ]


def test_dedup():
    code = "eval(a)\neval(b)\neval(c)"
    result = impl.solve(code)
    assert result == sorted(set(result))
    assert result == ["vuln:eval"]


def test_sorted_output():
    code = "pickle.loads(x)\neval(y)"
    result = impl.solve(code)
    assert result == sorted(result)


def test_no_false_positive_md5_variable():
    # "md5_value" does not contain "md5(" so should not trigger
    assert impl.solve("md5_value = compute_hash()") == []


def test_shell_false_not_flagged():
    # shell=False is not dangerous — should not trigger
    assert impl.solve("subprocess.run(cmd, shell=False)") == []


# ---- run every test above ------------------------------------------------
_results = []
for _name, _fn in list(globals().items()):
    if _name.startswith("test_") and callable(_fn):
        try:
            _fn()
            _results.append((_name, None))
        except Exception as _e:
            _results.append((_name, _e))
for _name, _e in _results:
    print(("PASS  " if _e is None else "FAIL  ") + _name + ("" if _e is None else f": {_e!r}"))
print(f"{sum(e is None for _, e in _results)}/{len(_results)} tests passed")
